# Pelatihan dan Pemetaan Model JST ke FPGA (PyTorch, 10 Kelas)

Notebook ini adalah **langkah 1–4 alur utama** (bagian Student C): melatih model, menentukan
format titik tetap, mengkuantisasi ke INT8, mengekspor bobot, membuat berkas memori `.hex` yang dibaca perangkat keras,
lalu menyiapkan citra uji beserta golden output untuk pengujian di papan.

| Aspek | Isi |
|---|---|
| Kerangka kerja | PyTorch |
| Tugas | Klasifikasi 10 kelas MNIST (digit 0–9) |
| Arsitektur | MLP **196 – 64 – 10**, satu lapisan tersembunyi, ReLU |
| Bilangan | INT8 titik tetap, notasi Qm.n dengan **m termasuk bit tanda** (konsisten dengan Bab 2) |
| Perangkat target | FPGA **Cyclone IV EP4CE6** (papan OMDAZZ), Quartus Prime Lite |

Prapemrosesan 14×14 memakai **rata-rata blok piksel (block averaging)**. Ini tahap prapemrosesan
data, **bukan** lapisan *pooling*, sehingga model tetap jaringan *feedforward* tanpa lapisan konvolusi.

**Status: model sudah dibekukan.** Hasil notebook ini disalin ke `export_final/` dan dijaga sidik jari
SHA-256 di `export_final/MANIFEST.md`. Seluruh pengujian, sampai papan fisik, memakai berkas di
`export_final/`. Tahap 1–6 **tidak perlu dijalankan ulang**: pelatihan ulang tidak menghasilkan bobot
yang sama persis, sehingga semua pengujian harus diulang. Tahap 7 dan Tahap 8 boleh dijalankan sendiri karena
hanya membaca `export_final/`, dan hasilnya identik dengan berkas yang sudah ada. Keluaran yang tersimpan di bawah adalah
hasil jalankan yang dipakai.

## Tahap 1 - Memuat MNIST dan prapemrosesan 14x14

In [1]:
import numpy as np, torch, torch.nn as nn, struct, os

SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED)

RAW = os.path.join('data', 'MNIST', 'raw')

def read_idx(path):
    with open(path, 'rb') as f:
        magic, = struct.unpack('>I', f.read(4))
        ndim = magic & 0xFF
        dims = struct.unpack('>' + 'I'*ndim, f.read(4*ndim))
        return np.frombuffer(f.read(), dtype=np.uint8).reshape(dims)

if not os.path.exists(os.path.join(RAW, 'train-images-idx3-ubyte')):
    from torchvision import datasets
    datasets.MNIST(root='./data', train=True,  download=True)
    datasets.MNIST(root='./data', train=False, download=True)

X_train_raw = read_idx(os.path.join(RAW, 'train-images-idx3-ubyte'))
y_train     = read_idx(os.path.join(RAW, 'train-labels-idx1-ubyte'))
X_test_raw  = read_idx(os.path.join(RAW, 't10k-images-idx3-ubyte'))
y_test      = read_idx(os.path.join(RAW, 't10k-labels-idx1-ubyte'))
print(f'MNIST: {len(X_train_raw)} citra latih, {len(X_test_raw)} citra uji, ukuran 28x28')

def block_average_14(X):
    """28x28 -> 14x14 lewat rata-rata blok 2x2, lalu normalisasi ke [0,1]."""
    Xd = X.reshape(-1, 14, 2, 14, 2).mean(axis=(2, 4))
    return (Xd.reshape(-1, 196) / 255.0).astype(np.float32)

X_train = block_average_14(X_train_raw)
X_test  = block_average_14(X_test_raw)
print(f'Setelah block averaging: {X_train.shape[1]} nilai masukan per citra, rentang [{X_test.min():.1f}, {X_test.max():.1f}]')

MNIST: 60000 citra latih, 10000 citra uji, ukuran 28x28
Setelah block averaging: 196 nilai masukan per citra, rentang [0.0, 1.0]


## Tahap 2 - Definisi model 196 - 64 - 10

Jumlah parameter model adalah **13.258**: 13.184 bobot (196×64 + 64×10) dan 74 bias (64 + 10).
Di perangkat keras, **13.184 bobot disimpan di memori M9K**, sedangkan 74 bias disimpan terpisah
sebagai ROM kecil (`rom_bias8.v`) di logika.

In [2]:
N_INPUT, N_HIDDEN, N_OUTPUT = 196, 64, 10

class MLP(nn.Module):
    def __init__(self, n_hidden=N_HIDDEN):
        super().__init__()
        self.fc1  = nn.Linear(N_INPUT, n_hidden)   # lapisan tersembunyi
        self.relu = nn.ReLU()                       # f(x) = max(0, x)
        self.fc2  = nn.Linear(n_hidden, N_OUTPUT)  # lapisan keluaran
    def forward(self, x):
        return self.fc2(self.relu(self.fc1(x)))

model = MLP()
n_param = sum(p.numel() for p in model.parameters())
print(model)
print(f'Jumlah parameter: {n_param} ({n_param/1024:.1f} KB bila disimpan INT8)')
print(f'Ukuran INT8     : {n_param*8/1024:.1f} Kbit, {n_param*8/(270*1024)*100:.1f}% dari 270 Kbit memori M9K pada Cyclone IV EP4CE6')

MLP(
  (fc1): Linear(in_features=196, out_features=64, bias=True)
  (relu): ReLU()
  (fc2): Linear(in_features=64, out_features=10, bias=True)
)
Jumlah parameter: 13258 (12.9 KB bila disimpan INT8)
Ukuran INT8     : 103.6 Kbit, 38.4% dari 270 Kbit memori M9K pada Cyclone IV EP4CE6


## Tahap 3 - Pelatihan (PyTorch, Adam + Cross-Entropy)

In [3]:
Xtr = torch.from_numpy(X_train); Ytr = torch.from_numpy(y_train.astype(np.int64))
Xte = torch.from_numpy(X_test);  Yte = torch.from_numpy(y_test.astype(np.int64))

opt, lossf = torch.optim.Adam(model.parameters(), lr=1e-3), nn.CrossEntropyLoss()
EPOCHS, BATCH = 25, 128

for ep in range(EPOCHS):
    model.train(); perm = torch.randperm(len(Xtr))
    for i in range(0, len(Xtr), BATCH):
        idx = perm[i:i+BATCH]
        opt.zero_grad(); loss = lossf(model(Xtr[idx]), Ytr[idx]); loss.backward(); opt.step()
    if (ep+1) % 5 == 0:
        model.eval()
        with torch.no_grad():
            acc = (model(Xte).argmax(1) == Yte).float().mean().item()
        print(f'  Epoch {ep+1:>2}/{EPOCHS} | loss {loss.item():.4f} | akurasi uji {acc*100:.2f}%')

model.eval()
with torch.no_grad():
    acc_fp32 = (model(Xte).argmax(1) == Yte).float().mean().item()
print(f'\nAkurasi model referensi FP32: {acc_fp32*100:.2f}%')

  Epoch  5/25 | loss 0.1659 | akurasi uji 95.03%
  Epoch 10/25 | loss 0.0496 | akurasi uji 96.44%
  Epoch 15/25 | loss 0.0206 | akurasi uji 97.06%
  Epoch 20/25 | loss 0.0651 | akurasi uji 97.30%
  Epoch 25/25 | loss 0.0328 | akurasi uji 97.45%

Akurasi model referensi FP32: 97.45%


## Tahap 4 - Analisis distribusi bobot dan penentuan format Qm.n

Konvensi yang dipakai (sesuai Bab 2 dan Shawahna et al., 2022): **Qm.n dengan m sudah termasuk
bit tanda**, sehingga m + n = 8 dan jangkauannya adalah [-2^(m-1), 2^(m-1) - 2^(-n)].
Nilai n dipilih sebesar mungkin selama tidak ada nilai yang terpotong.

In [4]:
W1 = model.fc1.weight.detach().numpy(); B1 = model.fc1.bias.detach().numpy()
W2 = model.fc2.weight.detach().numpy(); B2 = model.fc2.bias.detach().numpy()
with torch.no_grad():
    HID = model.relu(model.fc1(Xte)).numpy()

def n_maksimum(arr):
    """n terbesar yang tidak menyebabkan pemotongan pada INT8."""
    mx = float(np.abs(arr).max())
    for n in range(7, -1, -1):
        if round(mx * 2**n) <= 127:
            return n, mx
    return 0, mx

print(f'{"Tensor":<22}{"|maks|":>9}{"n maks":>9}{"format":>10}')
for nama, arr in [('Masukan (piksel)', X_test), ('Bobot lapisan 1', W1), ('Bobot lapisan 2', W2),
                  ('Bias lapisan 1', B1), ('Bias lapisan 2', B2), ('Aktivasi tersembunyi', HID)]:
    n, mx = n_maksimum(arr)
    print(f'{nama:<22}{mx:>9.3f}{n:>9}{"Q"+str(8-n)+"."+str(n):>10}')

Tensor                   |maks|   n maks    format
Masukan (piksel)          1.000        6      Q2.6
Bobot lapisan 1           2.251        5      Q3.5
Bobot lapisan 2           2.220        5      Q3.5
Bias lapisan 1            0.580        7      Q1.7
Bias lapisan 2            0.181        7      Q1.7
Aktivasi tersembunyi     10.121        3      Q5.3


Dari hasil di atas ditetapkan konfigurasi berikut. Bobot kedua lapisan disamakan formatnya
agar unit MAC tidak perlu membedakan penskalaan antar lapisan.

In [5]:
N_IN, N_W, N_B, N_H = 6, 5, 7, 3   # bit pecahan untuk masukan, bobot, bias, aktivasi
print(f'Masukan   : Q{8-N_IN}.{N_IN}')
print(f'Bobot     : Q{8-N_W}.{N_W}')
print(f'Bias      : Q{8-N_B}.{N_B}')
print(f'Aktivasi  : Q{8-N_H}.{N_H}')

def kuantisasi(a, n):
    """Kuantisasi linear ke INT8 dengan faktor skala 2^n (Persamaan 2.6 pada Bab 2)."""
    return np.clip(np.round(a * (2.0**n)), -128, 127).astype(np.int32)

W1q, B1q = kuantisasi(W1, N_W), kuantisasi(B1, N_B)
W2q, B2q = kuantisasi(W2, N_W), kuantisasi(B2, N_B)
Xq       = kuantisasi(X_test, N_IN)

for nama, asli, kuant, n in [('W1', W1, W1q, N_W), ('W2', W2, W2q, N_W)]:
    terpotong = (np.abs(np.round(asli * 2.0**n)) > 127).mean() * 100
    nol = (kuant == 0).mean() * 100
    print(f'{nama}: rentang INT8 [{kuant.min()}, {kuant.max()}] | terpotong {terpotong:.2f}% | bernilai nol {nol:.1f}%')

Masukan   : Q2.6
Bobot     : Q3.5
Bias      : Q1.7
Aktivasi  : Q5.3
W1: rentang INT8 [-72, 48] | terpotong 0.00% | bernilai nol 5.0%
W2: rentang INT8 [-71, 41] | terpotong 0.00% | bernilai nol 1.6%


## Tahap 5 - Simulasi jalur integer perangkat keras

Bagian ini meniru persis apa yang akan dikerjakan FPGA: perkalian dan akumulasi bilangan bulat,
ReLU melalui pemeriksaan tanda, dan penskalaan ulang melalui pergeseran bit. Tidak ada operasi
*floating-point* sama sekali. Keluaran akhir cukup dibandingkan dengan `argmax`, sehingga
lapisan keluaran tidak perlu diskalakan ulang.

In [6]:
def forward_integer(Xq, W1q, B1q, W2q, B2q):
    acc1 = Xq.astype(np.int64) @ W1q.T.astype(np.int64)      # skala 2^(N_IN+N_W)
    acc1 += (B1q.astype(np.int64) << (N_IN + N_W - N_B))     # penyelarasan skala bias
    acc1 = np.maximum(acc1, 0)                                # ReLU
    Hq   = np.clip(acc1 >> (N_IN + N_W - N_H), -128, 127)     # aktivasi -> INT8
    acc2 = Hq @ W2q.T.astype(np.int64)                        # skala 2^(N_H+N_W)
    acc2 += (B2q.astype(np.int64) << (N_H + N_W - N_B))
    return acc2, acc1

logit_int, acc1_all = forward_integer(Xq, W1q, B1q, W2q, B2q)
acc_int8 = (logit_int.argmax(1) == y_test).mean()

print(f'Akurasi FP32  : {acc_fp32*100:.2f}%')
print(f'Akurasi INT8  : {acc_int8*100:.2f}%')
print(f'Degradasi     : {(acc_fp32-acc_int8)*100:+.2f} poin persentase  (target proposal: < 3)')
print(f'\nAkumulator maksimum teramati: lapisan 1 = {int(np.abs(acc1_all).max())}, '
      f'lapisan 2 = {int(np.abs(logit_int).max())}')
print(f'Lebar akumulator minimum   : {int(np.ceil(np.log2(np.abs(acc1_all).max())))+1} bit, '
      f'jadi akumulator 32 bit sudah lebih dari cukup')

Akurasi FP32  : 97.45%
Akurasi INT8  : 97.45%
Degradasi     : +0.00 poin persentase  (target proposal: < 3)

Akumulator maksimum teramati: lapisan 1 = 20847, lapisan 2 = 9582
Lebar akumulator minimum   : 16 bit, jadi akumulator 32 bit sudah lebih dari cukup


## Tahap 6 - Ekspor bobot (format antara .coe)

Notebook ini mengekspor bobot, bias, dan satu citra contoh sebagai berkas `.coe`: nilai heksadesimal
komplemen dua 8 bit, satu nilai per baris. Berkas `.coe` di sini hanya **format antara**, bukan berkas
yang dibaca Quartus.

Tata letak bobot lapisan 1 disusun **per neuron (neuron-major)**: neuron 0 menempati alamat 0–195,
neuron 1 menempati 196–391, dan seterusnya. Dengan susunan ini pengendali cukup memakai satu
pencacah alamat yang naik berurutan.

Langkah sesudahnya ada di **Tahap 7**: berkas `.coe` dari `export_final/` diubah menjadi berkas `.hex`
8 bank yang dibaca Verilog lewat `$readmemh` dan tertanam di blok M9K saat kompilasi Quartus.

In [7]:
OUT = 'export'
os.makedirs(OUT, exist_ok=True)

def export_coe(nama, data):
    data = np.asarray(data).flatten().astype(int)
    with open(os.path.join(OUT, nama), 'w') as f:
        f.write('memory_initialization_radix=16;\n')
        f.write('memory_initialization_vector=\n')
        for i, v in enumerate(data):
            f.write(f'{((int(v) + 256) % 256):02X}{";" if i == len(data)-1 else ","}\n')
    print(f'  {nama:<24} {len(data):>6} nilai')

export_coe('weights_layer1.coe', W1q)   # 64 x 196, urut per neuron
export_coe('weights_layer2.coe', W2q)   # 10 x 64
export_coe('bias_layer1.coe',    B1q)
export_coe('bias_layer2.coe',    B2q)

# satu citra uji sebagai vektor verifikasi
idx_poc = 0
export_coe('input_sample.coe', Xq[idx_poc])

acc2_poc, acc1_poc = forward_integer(Xq[idx_poc:idx_poc+1], W1q, B1q, W2q, B2q)
prediksi = int(acc2_poc.argmax())

with open(os.path.join(OUT, 'golden_output.txt'), 'w') as f:
    f.write('=== GOLDEN OUTPUT - REFERENSI VERIFIKASI BIT-EXACT ===\n')
    f.write(f'Citra uji indeks       : {idx_poc}\n')
    f.write(f'Label sebenarnya       : {int(y_test[idx_poc])}\n')
    f.write(f'Prediksi model INT8    : {prediksi}\n\n')
    f.write('Akumulator lapisan keluaran (10 nilai, sebelum argmax):\n')
    for k, v in enumerate(acc2_poc[0]):
        f.write(f'  digit {k}: {int(v)}\n')

with open(os.path.join(OUT, 'quant_config.txt'), 'w') as f:
    f.write('=== KONFIGURASI KUANTISASI ===\n')
    f.write('Konvensi: Qm.n dengan m termasuk bit tanda, m + n = 8\n\n')
    for nama, n in [('Masukan', N_IN), ('Bobot', N_W), ('Bias', N_B), ('Aktivasi tersembunyi', N_H)]:
        f.write(f'{nama:<22}: Q{8-n}.{n}  (skala 2^{n})\n')
    f.write(f'\nArsitektur : {N_INPUT} - {N_HIDDEN} - {N_OUTPUT}, aktivasi ReLU\n')
    f.write(f'Akurasi FP32: {acc_fp32*100:.2f}%\n')
    f.write(f'Akurasi INT8: {acc_int8*100:.2f}%\n')
    f.write(f'Degradasi   : {(acc_fp32-acc_int8)*100:+.2f} poin persentase\n')

torch.save(model.state_dict(), os.path.join(OUT, 'model_mlp_196_64_10.pt'))
print(f'\nSelesai. Prediksi citra uji indeks {idx_poc}: {prediksi} (label sebenarnya {int(y_test[idx_poc])})')

  weights_layer1.coe        12544 nilai
  weights_layer2.coe          640 nilai
  bias_layer1.coe              64 nilai
  bias_layer2.coe              10 nilai
  input_sample.coe            196 nilai

Selesai. Prediksi citra uji indeks 0: 7 (label sebenarnya 7)


## Tahap 7 - Pembuatan berkas memori FPGA (.hex)

Pada tahap ini bobot dan bias INT8 yang sudah dibekukan di `export_final/` diubah menjadi berkas
yang langsung dibaca rancangan Verilog. Tujuannya menyusun bobot sesuai cara akselerator membaca
memori: bobot dibagi ke **8 bank**, satu bank untuk setiap lajur MAC, sehingga delapan neuron dapat
dihitung bersamaan.

Bank ke-j menyimpan neuron j, j+8, j+16, dan seterusnya. Dengan susunan ini kedelapan bank memakai
**alamat yang sama** pada setiap siklus, sehingga pengendali cukup memakai satu pencacah alamat.

Tahap ini **tidak bergantung pada Tahap 1–6**: sel di bawah membaca langsung dari `export_final/`,
sehingga dapat dijalankan sendiri tanpa melatih ulang model. Hasilnya ditulis ke folder `../mem/`:

| Berkas | Isi | Dipakai oleh |
|---|---|---|
| `P8/mem_w_b0.hex` … `mem_w_b7.hex` | bobot 8 bank, masing-masing 1.696 nilai | `mem_w.v` (memori M9K) |
| `mem_d.hex` | isi awal memori data: 196 piksel + 64 ruang aktivasi | `mem_d.v` |
| `rom_bias8.v` | ROM bias per kelompok 8 neuron, 10 entri × 64 bit | `top_papan.v` |

In [ ]:
import os, re
import numpy as np

COE = 'export_final'      # model beku
MEM = os.path.join('..', 'mem')

P = 8                                        # jumlah lajur MAC = jumlah bank
N_INPUT, N_HIDDEN, N_OUTPUT = 196, 64, 10

def baca_coe(path):
    """Baca berkas .coe (heksadesimal komplemen dua 8 bit) menjadi larik integer bertanda."""
    badan = open(path).read().split('memory_initialization_vector=')[1]
    a = np.array([int(t, 16) for t in re.findall(r'\b([0-9A-Fa-f]{2})\b', badan)], dtype=np.int64)
    return np.where(a >= 128, a - 256, a)

def tulis_hex(path, data, nibble):
    """Satu nilai heksadesimal per baris, format yang dibaca $readmemh."""
    masker = (1 << (nibble * 4)) - 1
    with open(path, 'w') as f:
        for v in np.asarray(data).flatten():
            f.write(format(int(v) & masker, f'0{nibble}x') + '\n')

W1 = baca_coe(os.path.join(COE, 'weights_layer1.coe')).reshape(N_HIDDEN, N_INPUT)
W2 = baca_coe(os.path.join(COE, 'weights_layer2.coe')).reshape(N_OUTPUT, N_HIDDEN)
B1 = baca_coe(os.path.join(COE, 'bias_layer1.coe'))
B2 = baca_coe(os.path.join(COE, 'bias_layer2.coe'))
X  = baca_coe(os.path.join(COE, 'input_sample.coe'))
print(f'Dibaca dari {COE}: W1 {W1.shape}, W2 {W2.shape}, B1 {B1.shape}, B2 {B2.shape}, citra {X.shape}')

Dibaca dari export_final: W1 (64, 196), W2 (10, 64), B1 (64,), B2 (10,), citra (196,)


In [ ]:
# ---- bobot: 8 bank, bank ke-j menyimpan neuron j, j+8, j+16, ... ----
nA = -(-N_HIDDEN // P)          # putaran lapisan 1 = 8
nB = -(-N_OUTPUT // P)          # putaran lapisan 2 = 2
per_bank = nA * N_INPUT + nB * N_HIDDEN

bank = [np.zeros(per_bank, dtype=np.int64) for _ in range(P)]
for j in range(P):
    for g in range(nA):                          # lapisan 1: alamat g*196 + i
        n = g * P + j
        if n < N_HIDDEN:
            bank[j][g*N_INPUT:(g+1)*N_INPUT] = W1[n]
    for g in range(nB):                          # lapisan 2: alamat nA*196 + g*64 + i
        n = g * P + j
        if n < N_OUTPUT:
            aw = nA*N_INPUT + g*N_HIDDEN
            bank[j][aw:aw+N_HIDDEN] = W2[n]

# pemeriksaan: setiap bobot harus ditemukan kembali di bank dan alamat yang benar
salah = sum(not np.array_equal(bank[n % P][(n//P)*N_INPUT:(n//P+1)*N_INPUT], W1[n]) for n in range(N_HIDDEN))
salah += sum(not np.array_equal(bank[n % P][nA*N_INPUT+(n//P)*N_HIDDEN:nA*N_INPUT+(n//P+1)*N_HIDDEN], W2[n])
             for n in range(N_OUTPUT))
assert salah == 0, f'tata letak salah pada {salah} neuron'

os.makedirs(os.path.join(MEM, f'P{P}'), exist_ok=True)
for j in range(P):
    tulis_hex(os.path.join(MEM, f'P{P}', f'mem_w_b{j}.hex'), bank[j], 2)

blok = -(-per_bank // 1024)                      # 1 blok M9K = 1024 x 8 bit
print(f'{P} bank x {per_bank} nilai, {blok} blok M9K per bank -> {blok*P} blok untuk bobot')
print(f'Lapisan 1: {nA} putaran x {N_INPUT}, lapisan 2: {nB} putaran x {N_HIDDEN}')
print('Tata letak terverifikasi: setiap bobot ditemukan kembali pada alamat yang benar.')

8 bank x 1696 nilai, 2 blok M9K per bank -> 16 blok untuk bobot
Lapisan 1: 8 putaran x 196, lapisan 2: 2 putaran x 64
Tata letak terverifikasi: setiap bobot ditemukan kembali pada alamat yang benar.


In [ ]:
# ---- memori data: 196 piksel citra contoh, lalu 64 tempat untuk aktivasi lapisan 1 ----
mem_d = np.concatenate([X, np.zeros(N_HIDDEN, dtype=np.int64)])
tulis_hex(os.path.join(MEM, 'mem_d.hex'), mem_d, 2)
print(f'mem_d.hex: {len(mem_d)} nilai (piksel 0..{N_INPUT-1}, aktivasi {N_INPUT}..{len(mem_d)-1})')

mem_d.hex: 260 nilai (piksel 0..195, aktivasi 196..259)


In [ ]:
# ---- ROM bias: dialamati per kelompok 8 neuron, keluar 8 x 8 bit sekaligus ----
b_kel = []
for g in range(nA):
    b_kel.append(('L1', g*P, list(B1[g*P:(g+1)*P])))
for g in range(nB):
    isi = list(B2[g*P:(g+1)*P])
    b_kel.append(('L2', g*P, isi + [0] * (P - len(isi))))   # sisa diisi nol, dibuang pengendali

def kata64(nilai):
    """Gabungkan 8 bias menjadi satu kata 64 bit: {bias7, ..., bias0}."""
    return ''.join(f'{int(v) & 0xFF:02X}' for v in reversed(nilai))

baris = []
for k, (lap, n0, nilai) in enumerate(b_kel):
    if lap == 'L2' and n0 + P > N_OUTPUT:
        ket = f'L2 neuron {n0},{n0+1} + {P-(N_OUTPUT-n0)} pengisi'
    else:
        ket = f'{lap} neuron {n0}..{n0+P-1}'
    baris.append(f"            4'd{k} : dout = 64'h{kata64(nilai)};   // {ket}  [{', '.join(str(int(v)) for v in nilai)}]")

kepala = '''// ==========================================================================
//  rom_bias8.v
//  ROM bias untuk konfigurasi P = 8, dibangkitkan dari
//  Training/export_final oleh notebook Training_MLP_PyTorch.ipynb (Tahap 7).
//
//  Pada P = 8, kedelapan lajur MAC membutuhkan bias DELAPAN neuron
//  BERURUTAN pada saat yang sama. Karena itu ROM ini dialamati per
//  KELOMPOK, bukan per neuron, dan mengeluarkan 8 x 8 bit sekaligus.
//
//  Cara ini jauh lebih murah daripada menyalin rom_bias 74 entri
//  sebanyak delapan kali: isinya hanya 10 entri x 64 bit.
//
//      kelompok 0..7 : lapisan 1, neuron 0..63
//      kelompok 8..9 : lapisan 2, neuron 0..9
//
//  Kelompok 9 hanya memuat 2 neuron yang sah (neuron 8 dan 9);
//  enam sisanya diisi nol dan hasilnya DIBUANG oleh pengendali,
//  bukan dipakai. Lihat topeng lajur di ctrl_fsm_p8.v.
// ==========================================================================
`default_nettype none

module rom_bias8 (
    input  wire [3:0]  kelompok,          // 0..9
    output reg  [63:0] dout               // {bias7, ..., bias0}
);

    always @(*) begin
        case (kelompok)
'''
ekor = '''            default: dout = 64'h0000000000000000;
        endcase
    end

endmodule

`default_nettype wire
'''
with open(os.path.join(MEM, 'rom_bias8.v'), 'w', newline='\n') as f:
    f.write(kepala + '\n'.join(baris) + '\n' + ekor)
print(f'rom_bias8.v: {len(b_kel)} kelompok x 64 bit')
print(baris[0].strip())

rom_bias8.v: 10 kelompok x 64 bit
4'd0 : dout = 64'h02F30F0D04FF03F2;   // L1 neuron 0..7  [-14, 3, -1, 4, 13, 15, -13, 2]


Berkas di `../mem/` inilah yang dibaca Quartus saat kompilasi: isinya tertanam di blok M9K dan ikut
masuk ke `papan.sof`. Langkah berikutnya, Tahap 8, menyiapkan citra uji dan golden output untuk papan.

## Tahap 8 - Persiapan citra uji dan golden output untuk papan

Pada tahap ini disiapkan 16 citra uji beserta kunci jawabannya (golden output) untuk menguji
akselerator di papan. Tujuannya agar hasil hitungan FPGA dapat dibandingkan langsung dengan Python:
untuk setiap citra, papan harus menghasilkan tebakan dan nilai akumulator yang sama persis.

Citra dipilih dengan aturan tetap, tanpa pilih-pilih: **10 citra** pertama untuk setiap angka 0–9,
ditambah **6 citra** dengan indeks terkecil yang belum terpakai. Citra yang salah ditebak tetap dipakai,
karena papan harus sama dengan Python termasuk saat keduanya salah.

Tahap ini juga **tidak bergantung pada Tahap 1–7**: ia membaca model beku di `export_final/` dan test set
MNIST, sehingga dapat dijalankan sendiri. Hasilnya ditulis ke folder `../uji/`:

| Berkas | Isi | Dipakai oleh |
|---|---|---|
| `citra_00.hex` … `citra_15.hex` | 16 citra uji INT8 (Q2.6) | bahan `citra_papan.hex` |
| `golden_uji.txt` | tebakan dan 10 nilai akumulator tiap citra | `jalankan_papan.py`, tabel tebakan Python di `top_papan.v` |
| `daftar_uji.txt` | daftar citra, label, dan tebakan | dokumentasi |
| `citra_papan.hex` | 16 citra digabung menjadi satu ROM | `top_papan.v` (ROM citra di FPGA) |

In [ ]:
import os, re, struct
import numpy as np

DIR_MODEL = 'export_final'
DIR_MNIST = os.path.join('data', 'MNIST', 'raw')
UJI       = os.path.join('..', 'uji')

N_INPUT, N_HIDDEN, N_OUTPUT = 196, 64, 10
N_IN = 6                                   # masukan Q2.6
GESER_BIAS_1, GESER_ACC_1 = 4, 8           # lapisan 1 (sama dengan Tahap 5)
GESER_BIAS_2 = 1                           # lapisan 2, akumulator tidak digeser
JUMLAH_UJI = 16

def baca_idx(path):
    with open(path, 'rb') as f:
        magic, = struct.unpack('>I', f.read(4))
        dims = struct.unpack('>' + 'I' * (magic & 0xFF), f.read(4 * (magic & 0xFF)))
        return np.frombuffer(f.read(), dtype=np.uint8).reshape(dims)

def baca_coe(path):
    badan = open(path).read().split('memory_initialization_vector=')[1]
    a = np.array([int(t, 16) for t in re.findall(r'\b([0-9A-Fa-f]{2})\b', badan)], dtype=np.int64)
    return np.where(a >= 128, a - 256, a)

W1 = baca_coe(os.path.join(DIR_MODEL, 'weights_layer1.coe')).reshape(N_HIDDEN, N_INPUT)
W2 = baca_coe(os.path.join(DIR_MODEL, 'weights_layer2.coe')).reshape(N_OUTPUT, N_HIDDEN)
B1 = baca_coe(os.path.join(DIR_MODEL, 'bias_layer1.coe'))
B2 = baca_coe(os.path.join(DIR_MODEL, 'bias_layer2.coe'))

# test set MNIST, diproses persis seperti Tahap 1 dan Tahap 4
Xraw = baca_idx(os.path.join(DIR_MNIST, 't10k-images-idx3-ubyte'))
y    = baca_idx(os.path.join(DIR_MNIST, 't10k-labels-idx1-ubyte'))
X14  = (Xraw.reshape(-1, 14, 2, 14, 2).mean(axis=(2, 4)).reshape(-1, N_INPUT) / 255.0).astype(np.float32)
Xq   = np.clip(np.round(X14 * 2.0**N_IN), -128, 127).astype(np.int64)

def inferensi(X):
    """Jalur integer sama dengan FPGA: ReLU sebelum geser, lalu saturasi."""
    acc1 = (B1 << GESER_BIAS_1) + X @ W1.T
    act1 = np.clip(np.maximum(acc1, 0) >> GESER_ACC_1, -128, 127)
    return (B2 << GESER_BIAS_2) + act1 @ W2.T

akurasi = (inferensi(Xq).argmax(axis=1) == y).mean() * 100
print(f'MNIST: {len(Xq)} citra uji | akurasi INT8 seluruh test set: {akurasi:.2f}%')

MNIST: 10000 citra uji | akurasi INT8 seluruh test set: 97.45%


In [ ]:
# ---- pemilihan 16 citra: kemunculan pertama tiap digit, lalu indeks terkecil yang belum terpakai ----
pilih = [int(np.argmax(y == d)) for d in range(10)]
i = 0
while len(pilih) < JUMLAH_UJI:
    if i not in pilih:
        pilih.append(i)
    i += 1
pilih.sort()

os.makedirs(UJI, exist_ok=True)
hasil = []
for urut, idx in enumerate(pilih):
    acc2 = inferensi(Xq[idx:idx+1])[0]
    prediksi, label = int(np.argmax(acc2)), int(y[idx])
    nama = f'citra_{urut:02d}.hex'
    with open(os.path.join(UJI, nama), 'w') as f:        # 196 piksel + 64 ruang aktivasi
        for v in Xq[idx]:
            f.write(f'{int(v) & 0xFF:02x}\n')
        f.write('00\n' * N_HIDDEN)
    hasil.append((urut, nama, idx, label, prediksi, [int(v) for v in acc2]))
    print(f'[{urut:2d}] {nama}  MNIST#{idx:<3} label {label}  prediksi {prediksi}  '
          f'{"benar" if prediksi == label else "SALAH"}')

benar = sum(h[3] == h[4] for h in hasil)
print(f'\nBenar {benar} dari {len(hasil)} citra uji')

[ 0] citra_00.hex  MNIST#0   label 7  prediksi 7  benar
[ 1] citra_01.hex  MNIST#1   label 2  prediksi 2  benar
[ 2] citra_02.hex  MNIST#2   label 1  prediksi 1  benar
[ 3] citra_03.hex  MNIST#3   label 0  prediksi 0  benar
[ 4] citra_04.hex  MNIST#4   label 4  prediksi 4  benar
[ 5] citra_05.hex  MNIST#5   label 1  prediksi 1  benar
[ 6] citra_06.hex  MNIST#6   label 4  prediksi 4  benar
[ 7] citra_07.hex  MNIST#7   label 9  prediksi 9  benar
[ 8] citra_08.hex  MNIST#8   label 5  prediksi 5  benar
[ 9] citra_09.hex  MNIST#9   label 9  prediksi 9  benar
[10] citra_10.hex  MNIST#10  label 0  prediksi 0  benar
[11] citra_11.hex  MNIST#11  label 6  prediksi 6  benar
[12] citra_12.hex  MNIST#12  label 9  prediksi 9  benar
[13] citra_13.hex  MNIST#13  label 0  prediksi 0  benar
[14] citra_14.hex  MNIST#18  label 3  prediksi 3  benar
[15] citra_15.hex  MNIST#61  label 8  prediksi 8  benar

Benar 16 dari 16 citra uji


In [ ]:
# ---- daftar uji dan golden output per citra ----
with open(os.path.join(UJI, 'daftar_uji.txt'), 'w') as f:
    f.write('=== DAFTAR CITRA UJI UNTUK PENGUJIAN DI PAPAN ===\n\n')
    f.write('Model      : Training/export_final (beku)\n')
    f.write(f'Sumber     : MNIST test set, {len(Xq)} citra\n')
    f.write('Praproses  : 28x28 -> 14x14 rerata blok 2x2, kuantisasi Q2.6\n')
    f.write(f'Akurasi INT8 seluruh test set : {akurasi:.2f}%\n\n')
    f.write('Aturan pemilihan (deterministik, tanpa pemilihan subjektif):\n')
    f.write('  - 10 citra: kemunculan pertama tiap digit 0..9\n')
    f.write('  -  6 citra: indeks terkecil yang belum terpakai\n\n')
    f.write(f"{'No':>3}  {'Berkas':<16}{'MNIST#':>8}{'Label':>7}{'Prediksi':>10}  Hasil\n")
    f.write('-' * 56 + '\n')
    for urut, nama, idx, label, prediksi, _ in hasil:
        f.write(f"{urut:>3}  {nama:<16}{idx:>8}{label:>7}{prediksi:>10}  {'benar' if label == prediksi else 'SALAH'}\n")
    f.write('-' * 56 + '\n')
    f.write(f'\nBenar {benar} dari {len(hasil)} citra uji.\n')
    f.write('\nCitra yang salah diprediksi SENGAJA TIDAK DIBUANG. Papan harus\n')
    f.write('menghasilkan keluaran yang sama dengan Python, termasuk saat\n')
    f.write('keduanya salah -- itu justru bukti implementasinya bit-exact.\n')

with open(os.path.join(UJI, 'golden_uji.txt'), 'w') as f:
    f.write('=== ACUAN EMAS PER CITRA UJI ===\n')
    f.write('Sepuluh akumulator lapisan keluaran sebelum argmax.\n')
    f.write('Papan harus menghasilkan angka yang sama persis.\n\n')
    for urut, nama, idx, label, prediksi, acc2 in hasil:
        f.write(f'--- {nama}  MNIST#{idx}  label {label}  prediksi {prediksi}\n')
        for k, v in enumerate(acc2):
            f.write(f'      digit {k}: {v}\n')
        f.write('\n')

print('Ditulis: daftar_uji.txt dan golden_uji.txt')
print(f'Contoh golden citra_00 (MNIST#{hasil[0][2]}): prediksi {hasil[0][4]}, akumulator {hasil[0][5]}')

Ditulis: daftar_uji.txt dan golden_uji.txt
Contoh golden citra_00 (MNIST#0): prediksi 7, akumulator [-1269, -2393, -586, 663, -3113, -831, -5163, 2063, -968, -484]


In [ ]:
# ---- gabungkan 16 citra menjadi satu ROM: citra ke-k di alamat k*256 .. k*256+255 ----
N_PIKSEL, JARAK = 196, 256
rom = []
for k in range(JUMLAH_UJI):
    baris = [b.strip() for b in open(os.path.join(UJI, f'citra_{k:02d}.hex')).read().splitlines() if b.strip()]
    assert len(baris) == 260, f'citra_{k:02d}.hex berisi {len(baris)} baris'
    rom += baris[:N_PIKSEL] + ['00'] * (JARAK - N_PIKSEL)   # sisa 60 alamat diisi nol

with open(os.path.join(UJI, 'citra_papan.hex'), 'w', newline='\n') as f:
    f.write('\n'.join(rom) + '\n')
print(f'citra_papan.hex: {len(rom)} alamat, {JUMLAH_UJI} citra x {JARAK} alamat')

citra_papan.hex: 4096 alamat, 16 citra x 256 alamat


Tebakan Python untuk ke-16 citra (kolom `prediksi` di atas) ditulis sebagai tabel di `top_papan.v`.
Di papan, tebakan ini ditampilkan di samping tebakan FPGA, misalnya `00.77`, dan LED1 menyala bila
keduanya sama. Nilai akumulator di `golden_uji.txt` dipakai simulasi `jalankan_papan.py` untuk
mencocokkan hasil FPGA sampai ke angkanya.

## Ringkasan dan langkah berikutnya

| Hasil | Nilai |
|---|---|
| Akurasi FP32 | 97,45% |
| Akurasi INT8 (jalur integer perangkat keras) | 97,45%, degradasi 0 poin |
| Format | masukan Q2.6, bobot Q3.5, bias Q1.7, aktivasi Q5.3 |
| Akumulator | 32 bit (yang terpakai minimal 16 bit) |

Keluaran Tahap 1–6 ditulis ke `export/`, lalu dibekukan sebagai `export_final/` (lihat `MANIFEST.md`).
Tahap 7 membaca `export_final/` dan menulis berkas memori FPGA ke `../mem/`: bobot `.hex` 8 bank,
`mem_d.hex`, dan `rom_bias8.v`. Tahap 8 menulis 16 citra uji, `citra_papan.hex`, dan golden output
`golden_uji.txt` ke `../uji/`; 16 dari 16 citra ditebak benar.